# Sesión 3 · Su primer modelo de Machine Learning
**Curso:** Uso de IA para el análisis descriptivo y predictivo en energías renovables

> Ejecute cada celda con `Shift + Enter` y lea la explicación antes de cada paso. Todos los datos son **sintéticos** (simulados) para practicar.

**Objetivo:** entrenar una regresión lineal para estimar la potencia de un panel solar, medir el error con datos no vistos y provocar un sobreajuste para reconocerlo.

## 1. Datos
Simulamos 120 días de datos horarios: irradiancia (W/m²), temperatura (°C), nubosidad y potencia (kW). La potencia es lo que queremos predecir.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)
horas = pd.date_range('2025-01-01', periods=24*120, freq='h')
h = horas.hour.values
sol = np.clip(np.sin((h-6)/12*np.pi), 0, None)            # ciclo diario del sol
nube = np.clip(rng.normal(0.3, 0.25, len(horas)), 0, 0.9)  # nubosidad
irr = 1000*sol*(1-nube)+rng.normal(0,15,len(horas))
irr = np.clip(irr, 0, None)
temp = 18 + 8*sol + rng.normal(0,1.2,len(horas))
pot = np.clip(0.18*irr*(1-0.004*(temp-25)) + rng.normal(0,2,len(horas)), 0, None)  # kW
df = pd.DataFrame({'irr':irr,'temp':temp,'nube':nube,'hora':h,'pot':pot}, index=horas)
df.head()

## 2. Mirar antes de modelar
Siempre graficamos primero. Debe verse que más irradiancia significa más potencia.

In [ ]:
df.plot.scatter(x='irr', y='pot', s=4, alpha=0.4)
plt.title('Potencia vs irradiancia')
plt.show()

## 3. Dividir en train y test por fecha
Entrenamos con el pasado (80 %) y probamos con el futuro (20 %). **No** mezclamos al azar porque es una serie de tiempo.

In [ ]:
corte = int(len(df)*0.8)
train, test = df.iloc[:corte], df.iloc[corte:]
print('Train:', len(train), '| Test:', len(test))

## 4. Línea base
Antes de un modelo, lo más simple: predecir siempre el promedio del entrenamiento. Cualquier modelo útil debe superarlo.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
base = np.full(len(test), train['pot'].mean())
print('MAE línea base (kW):', round(mean_absolute_error(test['pot'], base),2))

## 5. Regresión lineal con una variable
El modelo busca `pot = a × irr + b`. Con `fit` aprende a y b.

In [ ]:
from sklearn.linear_model import LinearRegression
m1 = LinearRegression().fit(train[['irr']], train['pot'])
print('a =', round(m1.coef_[0],3), '| b =', round(m1.intercept_,2))
pred1 = m1.predict(test[['irr']])

## 6. Medir el error
- **MAE**: promedio del error absoluto (kW).
- **RMSE**: castiga más los errores grandes.
- **R²**: 1 es perfecto, 0 es como predecir el promedio.

In [ ]:
def metricas(y, p):
    return {'MAE': round(mean_absolute_error(y,p),2), 'RMSE': round(mean_squared_error(y,p)**0.5,2), 'R2': round(r2_score(y,p),3)}
print('Irradiancia sola:', metricas(test['pot'], pred1))

## 7. Agregar más variables
Incluimos temperatura y nubosidad. ¿Mejora el error?

In [ ]:
cols = ['irr','temp','nube']
m2 = LinearRegression().fit(train[cols], train['pot'])
pred2 = m2.predict(test[cols])
print('Tres variables:', metricas(test['pot'], pred2))

## 8. Real contra predicho
Si el modelo es bueno, los puntos caen cerca de la diagonal.

In [ ]:
plt.scatter(test['pot'], pred2, s=4, alpha=0.4)
lim = [0, test['pot'].max()]
plt.plot(lim, lim, 'r--')
plt.xlabel('Real (kW)'); plt.ylabel('Predicho (kW)')
plt.show()

## 9. Provocar un sobreajuste
Un polinomio de grado alto sobre pocos datos memoriza el ruido. Usamos solo 40 puntos de entrenamiento y comparamos el error en train y en test.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
peq = train.sample(40, random_state=1)
for g in [1, 3, 12]:
    mod = make_pipeline(PolynomialFeatures(g), StandardScaler(), LinearRegression()).fit(peq[['irr']], peq['pot'])
    e_tr = mean_absolute_error(peq['pot'], mod.predict(peq[['irr']]))
    e_te = mean_absolute_error(test['pot'], mod.predict(test[['irr']]))
    print(f'Grado {g:2d} | MAE train {e_tr:6.2f} | MAE test {e_te:6.2f}')

**Cómo leerlo:** si el error en train es mucho menor que en test, el modelo memorizó (sobreajuste). Lo ideal es que ambos sean parecidos y bajos.

## 10. Ejercicios
1. Cambie el 80 % del corte por 70 % y observe el MAE.
2. Use solo `temp` como variable. ¿Es mejor o peor que `irr`?
3. Pruebe grados 2, 5 y 8 en el paso 9 y anote en qué grado empieza el sobreajuste.
4. Explique con sus palabras por qué no mezclamos al azar train y test.